# EMOS Tutorial: Channel Material Screening for MOSFET Design

This notebook turns the screening strategy into a minimal, executable workflow that starts with database extraction and ends with device-level ranking using the MOSFET Evaluation feature.

The example is intentionally pragmatic:
- Stage 1 uses a broad, lenient database query to maximize recall.
- Stage 2 uses GBFS to add consistent electronic descriptors.
- Stage 3 uses MatterSim as a relaxation-based stability proxy.
- Stage 4 uses SynthNN to remove candidates with low synthesis likelihood.
- Stage 5 ranks the survivors with the MOSFET solver using a fixed device stack.

The code is kept short on purpose. Increase the batch sizes later if you want a larger search.

In [ ]:
import os
import sys
import json
import hashlib
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from pymatgen.core import Structure

workspace_root = Path.cwd().resolve()
while workspace_root != workspace_root.parent:
    if (workspace_root / 'Features').is_dir() and (workspace_root / 'Information_Units').is_dir():
        break
    workspace_root = workspace_root.parent

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from Features.Materials_Exploration.DatabaseExtractor.DatabaseExtractorFeature import DatabaseExtractorFeature
from Features.Electronics_Application.MosfetEvaluator.MosfetEvaluatorFeature import MosfetEvaluatorFeature
from Information_Units.Predictors.Gbfs.GbfsClient import GbfsClient
from Information_Units.Predictors.Mattersim.MattersimPredictor import MattersimPredictor
from Information_Units.Predictors.Synthnn.SynthnnPredictor import SynthnnPredictor


class NotebookLogger:
    def __init__(self):
        self.messages = []

    def log(self, message, level='info'):
        self.messages.append((level, message))


logger = NotebookLogger()


def flatten_cifs(extraction_payload):
    rows = []
    for db_key, db_entry in extraction_payload.get('databases', {}).items():
        payload = db_entry.get('payload', {})
        for cif_text in payload.get('cif_strings', []) if isinstance(payload, dict) else []:
            rows.append({'source': db_key, 'cif': cif_text})
    return rows


def reduced_formula(cif_text):
    try:
        return Structure.from_str(cif_text, fmt='cif').composition.reduced_formula
    except Exception:
        return 'unparsed'


def dedupe_candidates(rows):
    seen = set()
    unique = []
    for row in rows:
        digest = hashlib.sha1(row['cif'].encode('utf-8')).hexdigest()
        if digest in seen:
            continue
        seen.add(digest)
        unique.append({**row, 'formula': reduced_formula(row['cif']), 'cif_sha1': digest})
    return unique


def force_norm_max(force_array):
    if force_array is None:
        return np.nan
    arr = np.asarray(force_array, dtype=float)
    if arr.size == 0:
        return np.nan
    if arr.ndim == 1:
        return float(np.abs(arr).max())
    return float(np.linalg.norm(arr, axis=1).max())


def print_stage_summary(stage_name, criteria_lines, input_count, output_count, input_unique=None, output_unique=None):
    print(f"\n=== {stage_name} Summary ===")
    print('Selector / Threshold Criteria:')
    for line in criteria_lines:
        print(f"- {line}")
    print(f"Input candidates : {input_count}")
    if input_unique is not None:
        print(f"Input unique formulas : {input_unique}")
    print(f"Output candidates: {output_count}")
    if output_unique is not None:
        print(f"Output unique formulas: {output_unique}")


print(f'Workspace root: {workspace_root}')

Workspace root: /home/soe/EMOS


## Optional Service Setup

GBFS and MatterSim are Docker-backed in this workflow. The cell below can start either service when needed; leave the flags disabled when the services are already running.

In [ ]:
START_GBFS = False
START_MATTERSIM = False

if START_GBFS:
    subprocess.run(['docker', 'compose', 'up', '-d', 'gbfs-pred'], cwd=workspace_root, check=False)
if START_MATTERSIM:
    subprocess.run(['docker', 'compose', 'up', '-d', 'mattersim'], cwd=workspace_root, check=False)

gbfs = GbfsClient('gbfs', logger=logger)
mattersim = MattersimPredictor('mattersim', logger=logger)
# Refresh health cache so notebook reflects current container state.
MattersimPredictor._health_cache = {'healthy': None, 'checked_at': 0.0}
synthnn = SynthnnPredictor('synthnn', logger=logger)
extractor = DatabaseExtractorFeature(logger=logger)
mosfet = MosfetEvaluatorFeature(logger=logger)

print('GBFS reachable:', gbfs.is_healthy())
print('MatterSim reachable:', mattersim.is_healthy())
print('SynthNN ready')
print('MOSFET evaluator ready')

GBFS ready
MatterSim reachable: True
SynthNN ready
MOSFET evaluator ready


/home/soe/EMOS/emos_env/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.7.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/soe/EMOS/emos_env/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.7.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/soe/EMOS/emos_env/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.7.1 when using version 1.8.0. Thi

## Stage 1: Broad Candidate Generation with Database Extractor

This stage follows the broad-first, lenient strategy from the Database Extractor application examples. We keep the query permissive: semiconducting materials with moderate band gap, low hull distance, and 2-3 elements.

This is intentionally a *pre-screen*, not the final materials list.

In [21]:
stage1_payload = {
    'batchSize': 120,
    'retrievalMode': 'lenient',
    'targetCompositions': '',
    'queryValues': {
        'band_gap': [0.6, 4.0],
        'hull_distance': [0.0, 0.25],
        'nelements': [2, 3],
    },
    'active_databases': [
        {'value': 'jarvisdft', 'name': 'JARVIS-DFT'},
        {'value': 'alexandria', 'name': 'Alexandria'},
    ],
}

stage1_outputs = extractor.process(stage1_payload)
stage1_rows = dedupe_candidates(flatten_cifs(stage1_outputs['extraction']))
stage1_df = pd.DataFrame(stage1_rows)

print('Stage 1 unique CIF candidates:', len(stage1_df))
print('Stage 1 unique formulas:', stage1_df['formula'].nunique())
display(stage1_df[['formula', 'source']].head(10))

print_stage_summary(
    stage_name='Stage 1 (Database Extraction)',
    criteria_lines=[
        "Retrieval mode = lenient",
        "Databases = JARVIS-DFT + Alexandria",
        "band_gap in [0.6, 4.0] eV",
        "hull_distance in [0.0, 0.25] eV/atom",
        "nelements in [2, 3]",
        "batchSize = 120",
    ],
    input_count='N/A (database search space)',
    output_count=len(stage1_df),
    output_unique=stage1_df['formula'].nunique(),
)

/home/soe/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/soe/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 8 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/soe/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 16 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/soe/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 1 fraction

Stage 1 unique CIF candidates: 120
Stage 1 unique formulas: 71


,formula,source
0,LiLuS2,alexandria
1,LiLuS2,alexandria
2,LiLuS2,alexandria
3,LiLuSe2,alexandria
4,LiLuSe2,alexandria
5,LiLuSe2,alexandria
6,LiLuTe2,alexandria
7,LiLuTe2,alexandria
8,LiMgAs,alexandria
9,LiMgH3,alexandria



=== Stage 1 (Database Extraction) Summary ===
Selector / Threshold Criteria:
- Retrieval mode = lenient
- Databases = JARVIS-DFT + Alexandria
- band_gap in [0.6, 4.0] eV
- hull_distance in [0.0, 0.25] eV/atom
- nelements in [2, 3]
- batchSize = 120
Input candidates : N/A (database search space)
Output candidates: 120
Output unique formulas: 71


## Stage 2: Descriptor Completion with GBFS

GBFS gives a consistent set of electronic descriptors from the CIFs. We use band gap as the main hard filter, keep dielectric as a soft electrostatics screen, and retain mobility predictions for the final device stage.

The `head(24)` limit keeps the tutorial runtime manageable while still demonstrating the workflow.

In [17]:
stage2_input = stage1_df.head(36).to_dict('records')
stage2_predictions = gbfs.predict([row['cif'] for row in stage2_input])['results']

stage2_rows = []
for row, pred in zip(stage2_input, stage2_predictions):
    if pred['status'] != 'ok':
        continue
    props = pred['properties']
    stage2_rows.append({
        **row,
        'bandgap_eV': props.get('bandgap'),
        'dielectric': props.get('dielectric'),
        'mob_n_cm2_Vs': props.get('mob_n'),
        'mob_p_cm2_Vs': props.get('mob_p'),
        'e_form_eV_atom': props.get('e_form'),
    })

stage2_df = pd.DataFrame(stage2_rows)
stage2_df = stage2_df[
    stage2_df['bandgap_eV'].between(0.9, 4.0)
    & (stage2_df['dielectric'] >= 3.0)
].copy()

# Keep one representative CIF per reduced formula to avoid over-weighting near-duplicates.
stage2_df = (
    stage2_df.sort_values(['formula', 'mob_n_cm2_Vs', 'bandgap_eV'], ascending=[True, False, True])
    .drop_duplicates(subset=['formula'], keep='first')
    .sort_values(['mob_n_cm2_Vs', 'bandgap_eV'], ascending=[False, True])
    .head(16)
)

print('Stage 2 retained candidates:', len(stage2_df))
print('Stage 2 unique formulas:', stage2_df['formula'].nunique())
display(stage2_df[['formula', 'bandgap_eV', 'dielectric', 'mob_n_cm2_Vs', 'mob_p_cm2_Vs']])

print_stage_summary(
    stage_name='Stage 2 (GBFS Descriptor Completion)',
    criteria_lines=[
        "Input subset = first 36 Stage-1 candidates (runtime control)",
        "GBFS status must be 'ok'",
        "bandgap_eV in [0.9, 4.0]",
        "dielectric >= 3.0",
        "Drop duplicates by reduced formula (keep highest mob_n, then lower bandgap)",
        "Final selector = top 16 by mob_n desc, bandgap asc",
    ],
    input_count=len(stage2_input),
    output_count=len(stage2_df),
    input_unique=pd.DataFrame(stage2_input)['formula'].nunique(),
    output_unique=stage2_df['formula'].nunique(),
)

/home/soe/EMOS/Information_Units/Predictors/Gbfs/GbfsPredictor.py:594: FutureWarning: get_structures is deprecated; use parse_structures in pymatgen.io.cif instead.
The only difference is that primitive defaults to False in the new parse_structures method.So parse_structures(primitive=True) is equivalent to the old behavior of get_structures().
  parsed = parser.get_structures(primitive=True)
/home/soe/EMOS/emos_env/lib/python3.12/site-packages/matminer/utils/data.py:367: UserWarning: No data available for velocity_of_sound for F
  all_values = [getattr(e, property_name) for e in Element]
/home/soe/EMOS/emos_env/lib/python3.12/site-packages/matminer/utils/data.py:367: UserWarning: No data available for velocity_of_sound for P
  all_values = [getattr(e, property_name) for e in Element]
/home/soe/EMOS/emos_env/lib/python3.12/site-packages/matminer/utils/data.py:367: UserWarning: No data available for velocity_of_sound for S
  all_values = [getattr(e, property_name) for e in Element]
/hom

Stage 2 retained candidates: 16
Stage 2 unique formulas: 16


/home/soe/EMOS/emos_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/home/soe/EMOS/emos_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/soe/EMOS/emos_env/lib/python3.12/site-packages/matminer/utils/data.py:367: UserWarning: No data available for coefficient_of_linear_thermal_expansion for H
  all_values = [getattr(e, property_name) for e in Element]
/home/soe/EMOS/emos_env/lib/python3.12/site-packages/matminer/utils/data.py:367: UserWarning: No data available for coefficient_of_linear_thermal_expansion for He
  all_values = [getattr(e, property_name) for e in Element]
/home/soe/EMOS/emos_env/lib/python3.12/site-packages/matminer/utils/data.py:367: UserWarning: No data available for coefficient_of_linear_thermal_expans

,formula,bandgap_eV,dielectric,mob_n_cm2_Vs,mob_p_cm2_Vs
18,LiMnF3,1.230252,7.830001,1822.190957,7.655151
9,LiMgH3,3.296077,12.892876,508.471163,27.320428
13,LiMgP,2.390320,12.999740,244.446652,47.314158
11,LiMgN,2.515869,12.150330,192.364007,19.371298
8,LiMgAs,1.660391,13.875996,149.009965,37.012446
34,LiNb13O33,2.480260,20.753175,139.881359,7.557477
31,LiMoS2,1.621183,11.859092,98.236519,23.874024
15,LiMgSb,1.676826,14.970030,97.429941,83.077506
32,LiMoSe2,1.406804,13.781028,97.342664,33.015919
0,LiLuS2,2.931876,26.799241,62.281669,38.679253



=== Stage 2 (GBFS Descriptor Completion) Summary ===
Selector / Threshold Criteria:
- Input subset = first 36 Stage-1 candidates (runtime control)
- GBFS status must be 'ok'
- bandgap_eV in [0.9, 4.0]
- dielectric >= 3.0
- Drop duplicates by reduced formula (keep highest mob_n, then lower bandgap)
- Final selector = top 16 by mob_n desc, bandgap asc
Input candidates : 36
Input unique formulas : 19
Output candidates: 16
Output unique formulas: 16


## Stage 3: MatterSim Relaxation-Based Stability Proxy

The current MatterSim wrapper does not expose a thermodynamic stability label directly, so this tutorial uses a conservative proxy: successful relaxation with low residual force. This is not equivalent to a full stability consensus, but it is a practical screening step with the current EMOS implementation.

In [18]:
if not mattersim.is_healthy():
    raise RuntimeError('MatterSim is not reachable. Start it with docker compose up -d mattersim and rerun this cell.')

stage3_input = stage2_df.to_dict('records')
stage3_predictions = mattersim.predict(
    [row['cif'] for row in stage3_input],
    compute_energy=True,
    compute_forces=True,
    compute_stress=False,
    relax=True,
    relax_atoms=True,
    relax_cell=True,
)['results']

stage3_rows = []
for row, pred in zip(stage3_input, stage3_predictions):
    props = pred.get('properties', {})
    max_force = force_norm_max(props.get('relaxed_forces') or props.get('forces'))
    stage3_rows.append({
        **row,
        'mattersim_status': pred.get('status'),
        'relaxed_energy_eV': props.get('relaxed_energy', props.get('energy')),
        'max_force_eV_A': max_force,
    })

stage3_df = pd.DataFrame(stage3_rows)
stage3_df = stage3_df[
    (stage3_df['mattersim_status'] == 'ok')
    & (stage3_df['max_force_eV_A'] <= 0.10)
].copy()
stage3_df = stage3_df.sort_values('max_force_eV_A').head(8)

print('Stage 3 retained candidates:', len(stage3_df))
display(stage3_df[['formula', 'bandgap_eV', 'mob_n_cm2_Vs', 'max_force_eV_A', 'relaxed_energy_eV']])

print_stage_summary(
    stage_name='Stage 3 (MatterSim Stability Proxy)',
    criteria_lines=[
        "MatterSim container must be healthy",
        "Prediction status == 'ok'",
        "Residual force threshold: max_force_eV_A <= 0.10",
        "Final selector = top 8 with lowest max_force_eV_A",
    ],
    input_count=len(stage3_input),
    output_count=len(stage3_df),
    input_unique=pd.DataFrame(stage3_input)['formula'].nunique(),
    output_unique=stage3_df['formula'].nunique(),
)

Stage 3 retained candidates: 8


,formula,bandgap_eV,mob_n_cm2_Vs,max_force_eV_A,relaxed_energy_eV
2,LiMgP,2.390320,244.446652,8.049661e-07,-11.114395
7,LiMgSb,1.676826,97.429941,1.616870e-06,-9.179125
4,LiMgAs,1.660391,149.009965,1.749554e-06,-10.301859
3,LiMgN,2.515869,192.364007,7.731682e-06,-13.725203
10,LiLuSe2,2.684044,56.912858,4.569469e-04,-20.149315
13,LiLuTe2,1.675679,23.675799,1.058133e-03,-17.692194
9,LiLuS2,2.931876,62.281669,1.789471e-03,-22.142363
11,LiMgI3,3.227253,49.352844,4.377467e-03,-26.761328



=== Stage 3 (MatterSim Stability Proxy) Summary ===
Selector / Threshold Criteria:
- MatterSim container must be healthy
- Prediction status == 'ok'
- Residual force threshold: max_force_eV_A <= 0.10
- Final selector = top 8 with lowest max_force_eV_A
Input candidates : 16
Input unique formulas : 16
Output candidates: 8
Output unique formulas: 8


## Stage 4: Synthesizability Filtering with SynthNN

A threshold of 0.70 matches the SynthNN default classification rule and gives a shortlist that is more experimentally realistic without being overly restrictive.

In [19]:
stage4_input = stage3_df.to_dict('records')
stage4_predictions = synthnn.predict([row['cif'] for row in stage4_input])['results']

stage4_rows = []
for row, pred in zip(stage4_input, stage4_predictions):
    props = pred.get('properties', {})
    stage4_rows.append({
        **row,
        'synthesizable': props.get('synthesizable'),
        'synthesizability_score': props.get('synthesizability_score'),
    })

stage4_df = pd.DataFrame(stage4_rows)
stage4_df = stage4_df[stage4_df['synthesizability_score'] >= 0.70].copy()
stage4_df = stage4_df.sort_values('synthesizability_score', ascending=False).head(5)

print('Stage 4 retained candidates:', len(stage4_df))
display(stage4_df[['formula', 'bandgap_eV', 'mob_n_cm2_Vs', 'synthesizability_score']])

print_stage_summary(
    stage_name='Stage 4 (SynthNN Synthesizability)',
    criteria_lines=[
        "SynthNN inference on all Stage-3 candidates",
        "synthesizability_score >= 0.70",
        "Final selector = top 5 by synthesizability_score desc",
    ],
    input_count=len(stage4_input),
    output_count=len(stage4_df),
    input_unique=pd.DataFrame(stage4_input)['formula'].nunique(),
    output_unique=stage4_df['formula'].nunique(),
)

Stage 4 retained candidates: 3


,formula,bandgap_eV,mob_n_cm2_Vs,synthesizability_score
3,LiMgN,2.515869,192.364007,0.9006
6,LiLuS2,2.931876,62.281669,0.8711
4,LiLuSe2,2.684044,56.912858,0.7295



=== Stage 4 (SynthNN Synthesizability) Summary ===
Selector / Threshold Criteria:
- SynthNN inference on all Stage-3 candidates
- synthesizability_score >= 0.70
- Final selector = top 5 by synthesizability_score desc
Input candidates : 8
Input unique formulas : 8
Output candidates: 3
Output unique formulas: 3


## Stage 5: Device-Level Ranking with MOSFET Evaluation

The MOSFET solver uses the candidate-specific channel descriptors where we have them now: band gap, relative permittivity, and carrier mobilities. Missing transport parameters are filled with fixed baseline values so that ranking is driven by the candidate-dependent properties rather than by inconsistent assumptions.

The ranking rule follows the strategy document: enforce a practical off-current and threshold window first, then prioritize `Ion/Ioff`, and finally use `Ion` as the secondary sort key.

In [20]:
BASE_MOSFET_INPUTS = {
    'channelLengthNm': 14,
    'sourceDrainLengthNm': 4,
    'oxideThicknessNm': 1.0,
    'channelThicknessNm': 4.0,
    'temperatureK': 300,
    'gateWorkFunctionEv': 3.65,
    'sdWorkFunctionEv': 0.0,
    'channelDopingCm3': -1e15,
    'sourceDrainDopingCm3': 1e20,
    'gateVoltageSweepStartV': 0.0,
    'gateVoltageSweepStopV': 0.7,
    'numberOfGatePoints': 14,
    'drainVoltageSweepStartV': 0.0,
    'drainVoltageSweepStopV': 0.7,
    'numberOfDrainPoints': 13,
    'channelNc': 2.8e25,
    'channelNv': 1.04e25,
    'channelXiEv': 4.05,
    'channelVsatN': 2e5,
    'channelVsatP': 2e5,
    'channelPowN': 2.0,
    'channelPowP': 1.0,
    'insulatorNc': 1.0,
    'insulatorNv': 1.0,
    'insulatorEpsRel': 3.9,
    'insulatorUn': 1e-3,
    'insulatorUp': 1e-3,
    'insulatorXiEv': 0.9,
    'insulatorEgEv': 9.0,
    'insulatorVsatN': 2e5,
    'insulatorVsatP': 2e5,
    'insulatorPowN': 2.0,
    'insulatorPowP': 1.0,
}

def build_mosfet_payload(row):
    # GBFS mobility is in cm^2/Vs; the solver expects m^2/Vs.
    mu_n = max(float(row['mob_n_cm2_Vs']) * 1e-4, 1e-5)
    mu_p = max(float(row['mob_p_cm2_Vs']) * 1e-4, 1e-5)
    return {
        **BASE_MOSFET_INPUTS,
        'channelEgEv': float(np.clip(row['bandgap_eV'], 0.8, 4.0)),
        'channelEpsRel': float(np.clip(row['dielectric'], 2.0, 40.0)),
        'channelUn': mu_n,
        'channelUp': mu_p,
    }

stage5_rows = []
for row in stage4_df.to_dict('records'):
    outputs = mosfet.process(build_mosfet_payload(row))
    metrics = outputs['key_metrics']
    stage5_rows.append({
        'formula': row['formula'],
        'bandgap_eV': row['bandgap_eV'],
        'mob_n_cm2_Vs': row['mob_n_cm2_Vs'],
        'synthesizability_score': row['synthesizability_score'],
        'Id_on_uA_per_um': metrics['Id_on_uA_per_um'],
        'Id_off_uA_per_um': metrics['Id_off_uA_per_um'],
        'Vth_approx_V': metrics['Vth_approx_V'],
        'Ion_Ioff_ratio': metrics['Ion_Ioff_ratio'],
    })

stage5_df = pd.DataFrame(stage5_rows)
stage5_df['passes_device_screen'] = (
    (stage5_df['Id_off_uA_per_um'] <= 1e-2)
    & stage5_df['Vth_approx_V'].between(0.15, 0.65)
)
stage5_df['Vth_distance'] = (stage5_df['Vth_approx_V'] - 0.35).abs()
stage5_ranked = stage5_df.sort_values(
    ['passes_device_screen', 'Ion_Ioff_ratio', 'Id_on_uA_per_um', 'Vth_distance'],
    ascending=[False, False, False, True],
).reset_index(drop=True)

display(stage5_ranked)

print_stage_summary(
    stage_name='Stage 5 (MOSFET Device Ranking)',
    criteria_lines=[
        "Use fixed MOSFET geometry/bias stack for all candidates",
        "Map candidate properties into channelEgEv, channelEpsRel, channelUn, channelUp",
        "Device screen: Id_off_uA_per_um <= 1e-2 and 0.15 <= Vth_approx_V <= 0.65",
        "Ranking selector: passes_device_screen desc, Ion_Ioff_ratio desc, Id_on_uA_per_um desc, Vth_distance asc",
    ],
    input_count=len(stage4_df),
    output_count=len(stage5_ranked),
    input_unique=stage4_df['formula'].nunique(),
    output_unique=stage5_ranked['formula'].nunique(),
)

,formula,bandgap_eV,mob_n_cm2_Vs,synthesizability_score,Id_on_uA_per_um,Id_off_uA_per_um,Vth_approx_V,Ion_Ioff_ratio,passes_device_screen,Vth_distance
0,LiMgN,2.515869,192.364007,0.9006,40.367008,0.000388,0.484615,104054.578898,True,0.134615
1,LiLuSe2,2.684044,56.912858,0.7295,48.439362,0.003596,0.484615,13470.267514,True,0.134615
2,LiLuS2,2.931876,62.281669,0.8711,60.875240,0.006078,0.484615,10015.132490,True,0.134615



=== Stage 5 (MOSFET Device Ranking) Summary ===
Selector / Threshold Criteria:
- Use fixed MOSFET geometry/bias stack for all candidates
- Map candidate properties into channelEgEv, channelEpsRel, channelUn, channelUp
- Device screen: Id_off_uA_per_um <= 1e-2 and 0.15 <= Vth_approx_V <= 0.65
- Ranking selector: passes_device_screen desc, Ion_Ioff_ratio desc, Id_on_uA_per_um desc, Vth_distance asc
Input candidates : 3
Input unique formulas : 3
Output candidates: 3
Output unique formulas: 3


## Notes

- This notebook is a tutorial-sized example, so each stage uses small candidate caps to keep runtime reasonable.
- Stage 3 is a relaxation-based proxy for stability, not a full thermodynamic stability analysis.
- Stage 5 uses the MOSFET evaluator as it exists today, so the final ranking is based on `Ion/Ioff`, `Ion`, and a practical `Vth` window rather than on SS or DIBL.
- Once EMOS exposes richer stability and device metrics, the same structure can be tightened without changing the notebook layout.